In [206]:
import pandas as pd

In [207]:
ALL_SUBJECTS = [
    "Agriculture", "Geography", "French", "Economics", "Biology", "Igbo", "History", "Physics", "Chemistry",
    "Further Mathematics", "Civic Education", "Computer Science", "Hausa", "English Language", "Government", "Mathematics",
    "Yoruba", "Literature in English"
]

In [ ]:
def load_students(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename)

    class_level_map = {
        10: "SS1",
        11: "SS2",
        12: "SS3"
    }

    #  Replaces all values. Any value for the column that's not mapped in the dictionary key, is replace with NaN (meaning an empty value)
    # df["class_level"] = df["class_level"].map(class_level_map)

    # Only replaces matched values. Does not attempt to replace values not provided in the dictionary key.
    df["class_level"] = df["class_level"].replace(class_level_map)

    df["total_score"] = df[ALL_SUBJECTS].mean(axis=1).round(1)

    return df

### Task 1

Group the student data by class level and calculate:
   - The average score for each subject
   - The number of students in each class level
   - The minimum and maximum scores in each subject

In [209]:
def class_statistics(df):
    print("\n" + "="*50)
    print("TASK 1: CLASS LEVEL STATISTICS")
    print("="*50)

    class_level_group_df = df.groupby("class_level")

    # Task 1.1: Average for each subject
    subjects_avg_score = class_level_group_df[ALL_SUBJECTS].mean().round(1).T

    print(subjects_avg_score)

    # Task 1.2: Number of students per class level
    print("\n--- Number of students per class level ---")
    student_counts = class_level_group_df["student_id"].count()
    student_counts.name = "num_students"

    print(student_counts)

    # Task 1.3: Min & Max scores for all subjects
    print("\n--- Min & max scores (for all subjects) by class level ---")
    min_max = class_level_group_df[ALL_SUBJECTS].agg(["min", "max"])
    print(min_max)

### Task 2

Identify which class level has the:
   - Highest average attendance
   - Best overall academic performance (across all subjects)
   - Largest gender performance gap in Mathematics

In [210]:
def class_performance_analysis(df):
    print("\n" + "="*50)
    print("TASK 2: CLASS PERFORMANCE ANALYSIS")
    print("="*50)

    class_level_group_df = df.groupby("class_level")

    # Task 2.1: Highest average attendance, by class level
    avg_attendance = class_level_group_df["attendance"].agg("mean").round(1)
    best_attendance = avg_attendance.idxmax()

    print(avg_attendance)
    print("\n")
    print("="*50)
    print(f"Highest average attendance: Class {best_attendance} ({avg_attendance[best_attendance]}%)")

    # Task 2.2: Best overall academic performance, by class level
    print("\n--- Average overall score by class level ---")
    avg_overall = class_level_group_df["total_score"].mean().round(2)
    best_academic_performance = avg_overall.idxmax()
    
    print(f"Best overall performance: Class {best_academic_performance} ({avg_overall[best_academic_performance]}% on avg)")

    # Task 2.3: Largest gender performance gap in Mathematics
    print("\n--- Largest gender performance gap in Mathematics ---")
    gender_maths = df.groupby(["class_level", "gender"])["Mathematics"].mean().round(2)

    print("Gender Maths")
    print(gender_maths)
    print(type(gender_maths))

    gender_maths_wide = gender_maths.unstack(level="gender")
    gender_maths_wide["gap"] = abs(gender_maths_wide["F"] - gender_maths_wide["M"]).round(2)

    biggest_gap = gender_maths_wide["gap"].idxmax()

    print("\n")
    print(gender_maths_wide)
    print(type(gender_maths_wide))
    print(f"\nLargest gender gap in Mathematics can be seen in {biggest_gap} (gap of {gender_maths_wide["gap"][biggest_gap]} points)")

### Task 3

Find out if there's a correlation between attendance and academic performance (hint: use groupby with multiple conditions)

In [211]:
def attendance_vs_performance(df: pd.DataFrame):
    print("\n" + "="*50)
    print("TASK 3: ATTENDANCE VS ACADEMIC PERFORMANCE")
    print("="*50)

    df["attendance_band"] = pd.cut(
        df["attendance"],
        bins=[0, 70, 85, 100],
        labels=["Low (<=70%)", "Medium (71-85%)", "High (>85%)"]
    )

    attendance_vs_perf = df.groupby(["class_level", "attendance_band"], observed=True)["total_score"].mean().round(2)

    print(attendance_vs_perf)

    correlation = round(df["attendance"].corr(df["total_score"]), 4)

    print(f"\nCorrelation (attendance vs overall score): {correlation}")
    print("(1.0 = perfect postive correlation/link | 0 = no link | -1.0 = negative link)")

### Task 4
Create a summary report showing the performance metrics for each class level

In [ ]:
def summary_report(df: pd.DataFrame):
    print("\n" + "="*50)
    print("TASK 4: SUMMARY REPORT (PER CLASS LEVEL)")
    print("="*50)

    summary = df.groupby("class_level").agg(
        num_students = ("student_id", "count"),
        avg_attendance = ("attendance", "mean"),
        avg_english = ("English Language", "mean"),
        avg_maths = ("Mathematics", "mean"),
        avg_overall = ("total_score", "mean"),
        min_overall = ("total_score", "min"),
        max_overall = ("total_score", "max")
    ).round(2)

    print(summary.to_string())

    summary.to_csv("../../data/students_report.csv")

In [213]:
def main():
    df = load_students("../../data/students.csv")

    summary_report(df)

In [214]:
main()


TASK 4: SUMMARY REPORT (PER CLASS LEVEL)
             num_students  avg_attendance  avg_english  avg_maths  avg_overall  min_overall  max_overall
class_level                                                                                             
SS1                   800           88.18        74.26      74.86        73.74         45.7         98.4
SS2                   800           88.09        72.58      73.15        73.17         43.1         94.9
SS3                   800           88.03        74.10      73.79        73.57         37.4         96.6
